# 01 · LangChain 环境冒烟测试

用于确认 `week23/.venv` 环境可用、模型可连通。

**模型来源由 `.env` 的 `LLM_PROVIDER` 决定，三选一**：

| 取值 | 说明 | 需要的变量 |
|---|---|---|
| `qwen` | 千问（类上用的网关，按 token 计费） | `QWEN_API_KEY`、`QWEN_BASE_URL`、`QWEN_MODEL` |
| `openai` | 任意 OpenAI 兼容接口（DeepSeek 等，计费） | `OPENAI_API_KEY`、`OPENAI_BASE_URL`、`OPENAI_MODEL` |
| `ollama` | 本机 Ollama（免费，断网可用） | `OLLAMA_MODEL` |

配置写在 `week23/.env`（已 gitignore），模板见 `.env.example`。

> Notebook 内核请选择 `week23/.venv`（Python 3.11.9）。

In [4]:
import sys
import importlib.metadata as md

print("Python     :", sys.version.split()[0])
print("Interpreter:", sys.executable)

for pkg in [
    "langchain",
    "langchain-core",
    "langchain-openai",
    "langchain-ollama",
    "langchain-community",
    "langchain-text-splitters",
    "langchain-chroma",
]:
    try:
        print(f"{pkg:26s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:26s} 未安装")

Python     : 3.11.9
Interpreter: d:\project\step1\week23\.venv\Scripts\python.exe
langchain                  1.4.7
langchain-core             1.7.0
langchain-openai           1.7.3
langchain-ollama           1.1.0
langchain-community        0.4.2
langchain-text-splitters   1.1.3
langchain-chroma           1.1.0


In [32]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_file = Path.cwd() / ".env"
# override=True：改完 .env 重跑本格就生效，不会被内核里的旧值挡住
loaded = load_dotenv(env_file, override=True)
print("找到 .env:", env_file.exists(), "| 是否加载:", loaded)
print("LLM_PROVIDER:", os.getenv("LLM_PROVIDER") or "(未设置)")

print("\n[千问]")
print("  QWEN_BASE_URL:", os.getenv("QWEN_BASE_URL") or "(未设置)")
print("  QWEN_MODEL   :", os.getenv("QWEN_MODEL") or "(未设置)")
print("  QWEN_API_KEY :", "已设置" if os.getenv("QWEN_API_KEY") else "(未设置)")

print("\n[OpenAI 兼容]")
print("  OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL") or "(未设置)")
print("  OPENAI_MODEL   :", os.getenv("OPENAI_MODEL") or "(未设置)")
print("  OPENAI_API_KEY :", "已设置" if os.getenv("OPENAI_API_KEY") else "(未设置)")

print("\n[本地 Ollama]")
print("  OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL") or "(未设置)")
print("  OLLAMA_MODEL   :", os.getenv("OLLAMA_MODEL") or "(未设置)")

print("\nOPENAI_MAX_TOKENS:", os.getenv("OPENAI_MAX_TOKENS") or "(未设置)")

找到 .env: True | 是否加载: True
LLM_PROVIDER: qwen

[千问]
  QWEN_BASE_URL: https://cn-beijing.maas.qianwenaiapi.com/compatible-mode/v1
  QWEN_MODEL   : qwen3.8-flash
  QWEN_API_KEY : 已设置

[OpenAI 兼容]
  OPENAI_BASE_URL: https://api.deepseek.com
  OPENAI_MODEL   : deepseek-flash
  OPENAI_API_KEY : 已设置

[本地 Ollama]
  OLLAMA_BASE_URL: http://127.0.0.1:11434
  OLLAMA_MODEL   : qwen3.5:2b

OPENAI_MAX_TOKENS: 512


In [33]:
elif PROVIDER == "qwen":
    from langchain_openai import ChatOpenAI

    api_key = os.getenv("QWEN_API_KEY")
    base_url = os.getenv("QWEN_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
    model_name = os.getenv("QWEN_MODEL", "qwen-plus")
    max_tokens = int(os.getenv("OPENAI_MAX_TOKENS", "1024"))
    thinking = (os.getenv("QWEN_ENABLE_THINKING") or "false").strip().lower() in {"1", "true", "yes", "on"}

    if not api_key:
        print("缺少 QWEN_API_KEY：请在 .env 里填写千问（阿里云百炼）的 API Key")
    else:
        llm = ChatOpenAI(
            model=model_name, base_url=base_url, api_key=api_key,
            temperature=0, max_tokens=max_tokens,
            # 开着思考时，reasoning token 会先把 max_tokens 吃光，导致正文为空
            extra_body={"enable_thinking": thinking},
        )
        print(
            f"已就绪：千问 | model = {model_name} | {base_url} | "
            f"输出上限 {max_tokens} tokens | thinking = {'开' if thinking else '关'}"
        )

已就绪：千问 | model = qwen3.8-flash | https://cn-beijing.maas.qianwenaiapi.com/compatible-mode/v1 | 单次输出上限 512 tokens


In [29]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位善于用比喻讲解技术概念的助教。"),
        ("human", "用一句话解释 {concept}。"),
    ]
)

# LCEL：提示词 | 模型 | 解析器
chain = prompt | llm | StrOutputParser()

if llm is None:
    print("跳过调用：尚未配置模型")
else:
    print(chain.invoke({"concept": "LangChain 的 LCEL"}))

LCEL 是 LangChain 的链式表达式语言，用类似 `prompt | model | parser` 的管道写法把提示、模型和输出解析等组件串成一条可运行的处理链。


In [15]:
if llm is None:
    print("跳过调用：尚未配置模型")
else:
    for chunk in chain.stream({"concept": "向量检索（RAG）"}):
        print(chunk, end="", flush=True)

向量检索就像给每篇资料贴上“意思坐标”的标签：你提问时，它不比对字面关键词，而是找与你问题“站在同一位置”的那几篇，塞给 AI 当参考——这就是 RAG 让模型从“闭卷硬背”变成“开卷翻书”的关键一步。

In [25]:
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

SYSTEM_PROMPT = "你是一位耐心、准确的技术助教，回答控制在三句话以内，必要时举个例子。"


class ChatSession:
    """最小可用的多轮对话助手：自己维护消息历史。

    多轮对话的本质不是模型「记得」，而是把历史消息每次一起发过去。
    """

    def __init__(self, model, system_prompt: str = SYSTEM_PROMPT, verbose: bool = True):
        self.model = model
        self.system = SystemMessage(content=system_prompt)
        self.messages: list[BaseMessage] = [self.system]
        self.verbose = verbose

    def chat(self, question: str) -> str:
        # 1) 用户这句入历史
        self.messages.append(HumanMessage(content=question))
        # 2) 整段历史一起发给模型（历史越长，这次请求越贵）
        reply = self.model.invoke(self.messages)
        # 3) 模型回复也必须写回历史，下一轮才「记得」
        self.messages.append(reply)
        if self.verbose:
            print(f"你：{question}")
            print(f"AI：{reply.content}\n")
        return reply.content

    def reset(self) -> None:
        """清空历史：既重置上下文，也把后续请求的 token 成本降回起点。"""
        self.messages = [self.system]
        print("（对话历史已清空，系统提示保留）")

    @property
    def turns(self) -> int:
        """已完成的问答轮数"""
        return (len(self.messages) - 1) // 2


# 省钱开关：False = 只定义类，一次请求都不发（默认关闭，需要时手动打开）
RUN_DEMO = False

if llm is None:
    print("跳过：尚未配置模型，请先运行上面的「读 .env」和「选模型」单元格")
elif not RUN_DEMO:
    print("已跳过演示调用（RUN_DEMO = False）：类已定义，手动调用 session.chat('...') 即可")
else:
    session = ChatSession(llm)

    # 第 1 轮：信息只在这一句里出现
    session.chat("我叫钱富森，正在学 LangChain 的 LCEL。")

    # 第 2 轮：问题本身不含任何个人信息，只能靠历史回答
    session.chat("我叫什么？正在学什么？")

    print("历史消息数：", len(session.messages), "| 已完成轮数：", session.turns)
    print("消息角色顺序：", [m.type for m in session.messages])

# —— 想自己手动连续对话，取消下面 6 行注释再运行 ——
# while True:
#     line = input("你（回车或输入 exit 结束）：").strip()
#     if line.lower() in {"", "exit", "quit"}:
#         break
#     session.chat(line)

已跳过演示调用（RUN_DEMO = False）：类已定义，手动调用 session.chat('...') 即可
